In [0]:
dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")


In [0]:
import json

import pyspark.sql.functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

from common_utils.logging import get_logger
from common_utils.generic_functions import (
    rename_columns,
    safe_cast
)


# ============================================================
# 0. LOGGER
# ============================================================

logger = get_logger("customers_silver")


# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

logger.info(
    "Reading Silver configuration from %s",
    config_path
)


# ============================================================
# 2. READ CONFIG
# ============================================================

with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 3. CONFIG SECTIONS
# ============================================================

source_config = config["source"]
target_config = config["target"]

source_table = source_config["source_table"]
source_name = source_config["source_name"]

target_table = target_config["target_table"]

rename_mapping = config.get(
    "rename_columns",
    {}
)

cast_mapping = config.get(
    "cast_columns",
    {}
)

drop_columns = config.get(
    "drop_columns",
    []
)

select_columns = config.get(
    "select_columns",
    [])

dedup_config = config.get(
    "deduplication",
    {}
)

merge_config = config.get(
    "merge",
    {}
)


# ============================================================
# 4. READ BRONZE
# ============================================================

logger.info(
    "[%s] Reading from Bronze table %s",
    source_name,
    source_table
)

df = spark.read.table(source_table)

logger.info(
    "[%s] Successfully read Bronze table",
    source_name
)


# ============================================================
# 5. RENAME COLUMNS
# ============================================================

logger.info(
    "[%s] Applying column renaming",
    source_name
)

df = rename_columns(
    df,
    rename_mapping
)


# ============================================================
# 6. CUSTOMER NAME TRANSFORMATION
# ============================================================

logger.info(
    "[%s] Transforming customer name",
    source_name
)

if "customer_name" in df.columns:

    df = (
        df
        .withColumn(
            "customer_name",
            F.lower(F.col("customer_name"))
        )
        .withColumn(
            "name_parts",
            F.split(
                F.col("customer_name"),
                ","
            )
        )
        .withColumn(
            "first_name",
            F.trim(
                F.expr("get(name_parts, 1)")
            )
        )
        .withColumn(
            "last_name",
            F.trim(
                F.expr("get(name_parts, 0)")
            )
        )
    )


# ============================================================
# 7. STANDARDISE CITY
# ============================================================

if "city" in df.columns:

    logger.info(
        "[%s] Standardising city",
        source_name
    )

    df = df.withColumn(
        "city",
        F.upper(F.col("city"))
    )


# ============================================================
# 8. CLEAN POSTCODE
# ============================================================

if "postcode" in df.columns:

    logger.info(
        "[%s] Cleaning postcode",
        source_name
    )

    df = df.withColumn(
        "postcode",
        F.regexp_replace(
            F.col("postcode"),
            r"\.0$",
            ""
        )
    )


# ============================================================
# 9. ADD COUNTRY
# ============================================================

logger.info(
    "[%s] Adding country",
    source_name
)

df = df.withColumn(
    "country",
    F.lit("USA")
)


# ============================================================
# 10. DROP UNNECESSARY COLUMNS
# ============================================================

logger.info(
    "[%s] Dropping unnecessary columns",
    source_name
)

existing_drop_columns = [
    column
    for column in drop_columns
    if column in df.columns
]

if existing_drop_columns:

    df = df.drop(
        *existing_drop_columns
    )

    logger.info(
        "[%s] Dropped columns: %s",
        source_name,
        existing_drop_columns
    )


# ============================================================
# 11. SAFE CAST
# ============================================================

logger.info(
    "[%s] Applying data type conversions",
    source_name
)

df = safe_cast(
    df,
    cast_mapping
)


# ============================================================
# 12. DEDUPLICATION
# ============================================================

if dedup_config.get("enabled", False):

    logger.info(
        "[%s] Removing duplicate records",
        source_name
    )

    key_columns = dedup_config["key_columns"]

    order_by_config = dedup_config["order_by"]

    order_expressions = []

    for order_column in order_by_config:

        column_name = order_column["column"]

        direction = order_column.get(
            "direction",
            "asc"
        )

        nulls = order_column.get(
            "nulls",
            "last"
        )

        column_expression = F.col(
            column_name
        )

        if direction.lower() == "desc":

            if nulls.lower() == "last":
                column_expression = (
                    column_expression.desc_nulls_last()
                )
            else:
                column_expression = (
                    column_expression.desc_nulls_first()
                )

        else:

            if nulls.lower() == "last":
                column_expression = (
                    column_expression.asc_nulls_last()
                )
            else:
                column_expression = (
                    column_expression.asc_nulls_first()
                )

        order_expressions.append(
            column_expression
        )

    window = (
        Window
        .partitionBy(*key_columns)
        .orderBy(*order_expressions)
    )

    df = (
        df
        .withColumn(
            "rn",
            F.row_number().over(window)
        )
        .filter(
            F.col("rn") == 1
        )
        .drop("rn")
    )

    logger.info(
        "[%s] Deduplication completed",
        source_name
    )


# ============================================================
# 13. FINAL COLUMN SELECTION
# ============================================================

logger.info(
    "[%s] Selecting final Silver columns",
    source_name
)

df = df.select(
    *select_columns
)


# ============================================================
# 14. CHECK TARGET TABLE
# ============================================================

logger.info(
    "[%s] Checking target table %s",
    source_name,
    target_table
)

if spark.catalog.tableExists(target_table):

    # ========================================================
    # SCD TYPE 1 MERGE
    # ========================================================

    logger.info(
        "[%s] Target exists - starting SCD Type 1 MERGE",
        source_name
    )

    target = DeltaTable.forName(
        spark,
        target_table
    )

    merge_keys = merge_config["key_columns"]

    merge_condition = " AND ".join(
        [
            f"target.{key} = source.{key}"
            for key in merge_keys
        ]
    )

    logger.info(
        "[%s] Merge condition: %s",
        source_name,
        merge_condition
    )

    (
        target.alias("target")
        .merge(
            df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    logger.info(
        "[%s] SCD Type 1 MERGE completed successfully",
        source_name
    )


else:

    # ========================================================
    # INITIAL LOAD
    # ========================================================

    logger.info(
        "[%s] Target table does not exist",
        source_name
    )

    logger.info(
        "[%s] Creating Silver schema",
        source_name
    )

    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS
        retaildataplatform.silver
    """)

    logger.info(
        "[%s] Writing initial data to %s",
        source_name,
        target_table
    )

    (
        df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    logger.info(
        "[%s] Silver table created successfully",
        source_name
    )


# ============================================================
# 15. COMPLETION
# ============================================================

logger.info(
    "[%s] Bronze to Silver pipeline completed successfully",
    source_name
)